# 02 - The Smallest Useful Eval: Router Accuracy

This notebook evaluates the intent router in isolation.

Evaluation scope:
- Load labeled router examples from `evaluation_dataset.csv`
- Evaluate the deterministic heuristic router
- Evaluate a harder challenging router dataset
- Optionally evaluate the LLM router if `OPENAI_API_KEY` is configured
- Compare accuracy, precision, recall, F1, confusion matrices, and failure cases


## Learning Goal

Start with the smallest useful automated eval: given a query, can the router choose the correct source? This lab teaches categorical labels, binary pass/fail scoring, failure tables, and why simple task-specific metrics are often more useful than broad quality scores.

## Where This Fits

Progression: data sanity -> router eval -> retrieval eval -> cascade eval -> fallback eval -> answer quality -> full benchmark -> ablation.

This is the first real behavior eval. It isolates one decision before retrieval, fallback, or answer generation can hide the source of the error.

## Related AI Evals Concepts

- Don't Use Generic Eval Metrics: route accuracy measures the actual behavior we care about.
- Don't Use Likert Scales: each row is correct or incorrect, not a vague 1-5 score.
- AI Eval Mistakes: perfect scores on easy examples are a signal to add harder cases.
- Error Analysis: inspect failed rows to decide what to improve next.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

PROJECT_ROOT


In [ ]:
import asyncio
import importlib
import os

import chromadb
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import agentic_rag.ingestion as ingestion_module  # noqa: E402
import agentic_rag.router as router_module  # noqa: E402
from agentic_rag.constants import SourceType  # noqa: E402

importlib.reload(ingestion_module)
importlib.reload(router_module)

from agentic_rag.ingestion import ensure_chroma_collections  # noqa: E402
from agentic_rag.llm import OpenAITextGenerator  # noqa: E402
from agentic_rag.router import QueryRouter  # noqa: E402
from agentic_rag.settings import Settings  # noqa: E402
from agentic_rag.telemetry import configure_tracing  # noqa: E402

pd.set_option("display.max_colwidth", 180)


In [ ]:
TRACE_DIR = PROJECT_ROOT / "otel_traces"
TRACE_DIR.mkdir(exist_ok=True)
TRACE_FILE = TRACE_DIR / "02_router_evaluation.jsonl"

trace_settings = Settings(
    _env_file=None,
    OTEL_TRACING_ENABLED=True,
    OTEL_TRACES_EXPORTER="file",
    OTEL_TRACES_FILE=TRACE_FILE,
    OTEL_SERVICE_NAME="agentic-rag-notebooks",
)
configure_tracing(trace_settings)

TRACE_FILE


## Load Base And Challenging Test Sets

The base dataset contains mostly clear examples. The challenging dataset contains ambiguous, multi-intent, recency-sensitive, and source-selection examples.


In [ ]:
base_eval_path = PROJECT_ROOT / "datasets/evaluation_dataset.csv"
challenge_eval_path = PROJECT_ROOT / "datasets/challenging_router_evaluation_dataset.csv"

base_df = pd.read_csv(base_eval_path)
challenge_df = pd.read_csv(challenge_eval_path)

if "Query" in base_df.columns:
    base_df = base_df.rename(columns={"Query": "query", "Expected_Source_Type": "expected_source_type"})

base_df["expected_source_type"] = base_df["expected_source_type"].astype(str)
challenge_df["expected_source_type"] = challenge_df["expected_source_type"].astype(str)

base_df.head()


In [ ]:
challenge_df.head()


In [ ]:
qna_df = pd.read_csv(PROJECT_ROOT / "datasets/medical_qna_dataset.csv")
device_df = pd.read_csv(PROJECT_ROOT / "datasets/medical_device_manuals_dataset.csv")

settings = Settings(_env_file=None, chroma_path=PROJECT_ROOT / "chroma_db")
client = chromadb.PersistentClient(path=str(settings.chroma_path))

collection_summary = ensure_chroma_collections(client, qna_df, device_df)
collection_summary


In [ ]:
dataset_balance = pd.concat(
    [
        base_df["expected_source_type"].value_counts().rename("base"),
        challenge_df["expected_source_type"].value_counts().rename("challenging"),
    ],
    axis=1,
).fillna(0).astype(int)

dataset_balance


## Define A Small, Verifiable Router Eval


In [ ]:
LABELS = [source.value for source in SourceType]


async def evaluate_router(router: QueryRouter, df: pd.DataFrame) -> pd.DataFrame:
    async def route_row(query: str) -> str:
        return (await router.route(query)).value

    predictions = await asyncio.gather(*(route_row(query) for query in df["query"].tolist()))
    results = df.copy()
    results["predicted_source_type"] = predictions
    results["route_correct"] = results["expected_source_type"] == results["predicted_source_type"]
    return results


def router_summary(results: pd.DataFrame) -> dict:
    expected = results["expected_source_type"]
    predicted = results["predicted_source_type"]
    return {
        "num_examples": len(results),
        "accuracy": accuracy_score(expected, predicted),
        "classification_report": classification_report(
            expected,
            predicted,
            labels=LABELS,
            zero_division=0,
            output_dict=True,
        ),
        "confusion_matrix": confusion_matrix(expected, predicted, labels=LABELS),
    }


def report_table(summary: dict) -> pd.DataFrame:
    report = pd.DataFrame(summary["classification_report"]).T
    return report.loc[[*LABELS, "accuracy", "macro avg", "weighted avg"]]


def confusion_matrix_df(summary: dict) -> pd.DataFrame:
    return pd.DataFrame(summary["confusion_matrix"], index=LABELS, columns=LABELS)


def failure_table(results: pd.DataFrame) -> pd.DataFrame:
    columns = ["query", "expected_source_type", "predicted_source_type"]
    optional_columns = ["category", "rationale"]
    columns.extend(column for column in optional_columns if column in results.columns)
    return results.loc[~results["route_correct"], columns]


def failure_mode_summary(results: pd.DataFrame) -> pd.DataFrame:
    failures = results.loc[~results["route_correct"]].copy()
    if failures.empty:
        return pd.DataFrame(columns=["failure_mode", "failures"])

    if "category" in failures.columns:
        summary = (
            failures.groupby("category", dropna=False)
            .size()
            .reset_index(name="failures")
            .rename(columns={"category": "failure_mode"})
        )
    else:
        summary = (
            failures.groupby(["expected_source_type", "predicted_source_type"], dropna=False)
            .size()
            .reset_index(name="failures")
        )
        summary["failure_mode"] = summary["expected_source_type"] + " -> " + summary["predicted_source_type"]
        summary = summary[["failure_mode", "failures"]]

    return summary.sort_values("failures", ascending=False).reset_index(drop=True)


def route_confusion_summary(results: pd.DataFrame) -> pd.DataFrame:
    failures = results.loc[~results["route_correct"]]
    if failures.empty:
        return pd.DataFrame(columns=["expected_source_type", "predicted_source_type", "failures"])
    return (
        failures.groupby(["expected_source_type", "predicted_source_type"], dropna=False)
        .size()
        .reset_index(name="failures")
        .sort_values("failures", ascending=False)
        .reset_index(drop=True)
    )


## Start With Clear Examples

This deterministic baseline is cheap and suitable for quick checks.


In [ ]:
heuristic_router = QueryRouter(mode="heuristic")

base_results = await evaluate_router(heuristic_router, base_df)
base_summary = router_summary(base_results)

base_summary["accuracy"]


In [ ]:
report_table(base_summary)


In [ ]:
confusion_matrix_df(base_summary)


In [ ]:
failure_table(base_results)


## Add Hard Examples To Avoid Saturation

These examples intentionally stress the router. A large drop from the base score means the router handles obvious examples but struggles with ambiguity, mixed intent, or recency-sensitive phrasing.


In [ ]:
challenge_results = await evaluate_router(heuristic_router, challenge_df)
challenge_summary = router_summary(challenge_results)

challenge_summary["accuracy"]


In [ ]:
report_table(challenge_summary)


In [ ]:
confusion_matrix_df(challenge_summary)


In [ ]:
failure_table(challenge_results)


## Group And Count Failure Modes

Raw failure rows are useful, but error analysis becomes more actionable when similar failures are grouped and counted. This mirrors the AI Evals workflow: inspect failed traces, assign or reuse failure categories, then prioritize the most frequent patterns.


In [ ]:
failure_mode_summary(challenge_results)


In [ ]:
route_confusion_summary(challenge_results)


In [ ]:
failure_table(challenge_results)


## Compare Easy And Hard Slices


In [ ]:
router_comparison = pd.DataFrame(
    [
        {
            "dataset": "base",
            "examples": len(base_results),
            "accuracy": base_summary["accuracy"],
            "failures": int((~base_results["route_correct"]).sum()),
        },
        {
            "dataset": "challenging",
            "examples": len(challenge_results),
            "accuracy": challenge_summary["accuracy"],
            "failures": int((~challenge_results["route_correct"]).sum()),
        },
    ]
)

router_comparison


## Pay Attention To

- Router accuracy is useful because it is scoped to one concrete decision.
- The base dataset can look solved while the challenging dataset still reveals failures.
- Failure tables are more useful than aggregate accuracy when deciding what to fix.
- Optional LLM-router comparisons should come after the offline eval is understood, because they add cost, latency, and model variability.


## Optional Advanced Path: LLM Router Evaluation

This section is disabled by default because it makes API calls. Set `RUN_LLM_ROUTER = True` to compare the LLM router against the same base and challenging datasets.


In [ ]:
RUN_LLM_ROUTER = True
LLM_ROUTER_MODEL = os.getenv("OPENAI_ROUTER_MODEL", "gpt-5-nano")
LLM_ROUTER_TIMEOUT = float(os.getenv("OPENAI_TIMEOUT", "60"))

llm_base_results = None
llm_challenge_results = None
llm_base_summary = None
llm_challenge_summary = None

if RUN_LLM_ROUTER:
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise RuntimeError("OPENAI_API_KEY must be set in the active environment for LLM router evaluation")

    generator = OpenAITextGenerator(
        api_key=api_key,
        model=LLM_ROUTER_MODEL,
        timeout=LLM_ROUTER_TIMEOUT,
    )
    llm_router = QueryRouter(generator=generator, mode="llm")

    llm_base_results = await evaluate_router(llm_router, base_df)
    llm_challenge_results = await evaluate_router(llm_router, challenge_df)
    llm_base_results["router_mode"] = "llm"
    llm_challenge_results["router_mode"] = "llm"
    llm_base_summary = router_summary(llm_base_results)
    llm_challenge_summary = router_summary(llm_challenge_results)

llm_base_summary, llm_challenge_summary


In [ ]:
if llm_base_summary is not None and llm_challenge_summary is not None:
    display(
        pd.DataFrame(
            [
                {"dataset": "base", "accuracy": llm_base_summary["accuracy"]},
                {"dataset": "challenging", "accuracy": llm_challenge_summary["accuracy"]},
            ]
        )
    )
    display(report_table(llm_base_summary))
    display(confusion_matrix_df(llm_base_summary))
    display(failure_table(llm_base_results))
    display(report_table(llm_challenge_summary))
    display(confusion_matrix_df(llm_challenge_summary))
    display(failure_table(llm_challenge_results))


## Export Route-Level Artifacts

Export the heuristic router outputs so downstream notebooks can inspect route-level behavior without recomputing it.


In [ ]:
base_output_path = PROJECT_ROOT / "output/router_evaluation_results.csv"
challenge_output_path = PROJECT_ROOT / "output/challenging_router_evaluation_results.csv"

base_results.to_csv(base_output_path, index=False)
challenge_results.to_csv(challenge_output_path, index=False)

base_output_path, challenge_output_path
